# Tomography Globe — 3D Printable Hollow Hemispheres

This notebook produces two OBJ hemisphere files suitable for full-colour 3D printing.
It uses ETOPO topography for surface displacement and a seismic tomography depth
slice for vertex colouring.

The workflow uses `create_hollow_hemispheres`, which:
1. Splits the displaced outer shell at the equator (with a capped plane cut).
2. Boolean-subtracts the smooth inner sphere from each half.
3. Returns two **watertight, manifold** hollow hemispheres.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from globe3d import (
    generate_sphere_points_fibonacci,
    load_netcdf_grid,
    calculate_displacement_scale,
    displace_vertices,
    displace_near_lines,
    displace_by_polygons,
    assign_vertex_colors,
    create_hollow_hemispheres,
    write_obj_with_vertex_colors
)

## 1. Model Parameters

In [ ]:
# --- Globe geometry ---
outer_points = 1000000   # number of vertices in the outer shell
inner_points = 20000     # number of vertices in the inner shell
model_radius_mm = 40.0   # 80 mm diameter globe
inner_scale = 0.8        # inner void radius as a fraction of the outer

# --- Displacement ---
vert_exagg = 50          # vertical exaggeration factor for topography
earth_radius_km = 6371.0
tomography_displacement_scale = -1.5
displace_inner_with_tomo = True
coastline_step_mm = 0.5

# --- Boolean engine ---
# 'manifold' (recommended, fast) or 'blender' (requires Blender installed)
boolean_engine = 'manifold'

## 2. Generate Base Spheres

In [ ]:
print("Generating outer sphere...")
outer_vertices, outer_faces = generate_sphere_points_fibonacci(outer_points, model_radius_mm)
print(f"  Outer: {len(outer_vertices):,} vertices, {len(outer_faces):,} faces")

print("Generating inner sphere...")
inner_vertices, inner_faces = generate_sphere_points_fibonacci(inner_points, model_radius_mm * inner_scale)
print(f"  Inner: {len(inner_vertices):,} vertices, {len(inner_faces):,} faces")

## 3. Load Geographic Grids

In [ ]:
dem_grid = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"
colour_grid = "../inputs/s40_depth_slice_2850.grd"

print("Loading ETOPO topography grid...")
topo_lats, topo_lons, topo_grid = load_netcdf_grid(dem_grid, lat_var='lat', lon_var='lon', data_var='z')
print(f"  Shape: {topo_grid.shape}")
print(f"  Latitude range: {topo_lats.min()} - {topo_lats.max()}")
print(f"  Longitude range: {topo_lons.min()} - {topo_lons.max()}")
print(f"  Z range: {topo_grid.min()} - {topo_grid.max()}")

print("Loading tomography grid...")
tomo_lats, tomo_lons, tomo_grid = load_netcdf_grid(colour_grid, lat_var='y', lon_var='x', data_var='z')
print(f"  Shape: {tomo_grid.shape}")
print(f"  Latitude range: {tomo_lats.min()} - {tomo_lats.max()}")
print(f"  Longitude range: {tomo_lons.min()} - {tomo_lons.max()}")
print(f"  Z range: {np.nanmin(tomo_grid)} - {np.nanmax(tomo_grid)}")

## 4. Displace Vertices

Offset the vertices radially according to the grids and geographic context provided (in this example, a surface topography model from ETOPO, and a slice from the S40RTS tomography model, as per above)

Retrieve the coastline shapefiles from:
[Natural Earth Downloads](https://www.naturalearthdata.com/downloads/110m-physical-vectors/)
(e.g., download `ne_110m_land.zip` and extract to `../inputs/coastlines/`).


In [ ]:
coastline_shp = "../inputs/coastlines/ne_110m_land.shp"

scale = calculate_displacement_scale(model_radius_mm, earth_radius_km, vertical_exagg=vert_exagg) # note this assumes a model in mm, the radius in km, and that the grid is in meters!
print(f"Displacement scale factor: {scale:.6e}")

print("Displacing outer vertices with topography...")
outer_vertices = displace_vertices(outer_vertices, topo_lats, topo_lons, topo_grid, scale, show_progress=True)

print("Displacing outer vertices with tomography...")
outer_vertices = displace_vertices(outer_vertices, tomo_lats, tomo_lons, tomo_grid, tomography_displacement_scale, show_progress=True)

if displace_inner_with_tomo:
    print("Displacing interior vertices with tomography...")
    inner_vertices = displace_vertices(inner_vertices, tomo_lats, tomo_lons, tomo_grid, tomography_displacement_scale*inner_scale, show_progress=True)

print("Applying 1mm step at the coastlines using shapefile...")
outer_vertices = displace_by_polygons(outer_vertices, coastline_shp, displacement=coastline_step_mm)


## 5. Split & Hollow into Hemispheres

`create_hollow_hemispheres` performs the correct order of operations:
1. Splits the displaced outer globe at the equator with a capped plane cut.
2. Boolean-subtracts the smooth inner sphere from each half.

The `manifold` boolean engine creates its own triangulation for the annular
cap (the flat ring between the outer and inner shells), so no additional
cap refinement is needed.

The result is two **watertight, manifold** hollow hemispheres — ready for slicing.

In [ ]:
print("Splitting and hollowing hemispheres...")
print(f"  Boolean engine: {boolean_engine}")

top_half, bottom_half = create_hollow_hemispheres(
    outer_vertices, outer_faces,
    inner_vertices, inner_faces,
    engine=boolean_engine,
)

if top_half is not None and bottom_half is not None:
    print(f"  Top:    {len(top_half.vertices):,} verts, {len(top_half.faces):,} faces, "
          f"watertight={top_half.is_watertight}, volume={top_half.volume:.1f} mm³")
    print(f"  Bottom: {len(bottom_half.vertices):,} verts, {len(bottom_half.faces):,} faces, "
          f"watertight={bottom_half.is_watertight}, volume={bottom_half.volume:.1f} mm³")
else:
    print("ERROR: Hemisphere creation failed.")

## 6. Colour Options

In [ ]:
# --- Colouring Options ---
# Option 1: Colour according to boundaries
cmap_bounds = mcolors.ListedColormap(['red', 'white', 'blue'])
norm = mcolors.BoundaryNorm([-10, -0.5, 0.5, 10], cmap_bounds.N)
cmap = cmap_bounds
colour_kwargs = {'norm': norm}

# Option 2: Colour using a continuous matplotlib cmap
# cmap = 'RdBu'
# colour_kwargs = {'vmin': -1, 'vmax': 1}

# Option 3: Colour using a discretised version of a matplotlib cmap
# cmap = plt.get_cmap('RdBu', 7)
# colour_kwargs = {'vmin': -2, 'vmax': 2}

## 7. Preview Colour Map

In [ ]:
plt.figure(figsize=(10, 5))
lon_grid, lat_grid = np.meshgrid(tomo_lons, tomo_lats)
plt.pcolormesh(lon_grid, lat_grid, tomo_grid, cmap=cmap, **colour_kwargs, shading='auto')
plt.colorbar(label='Value')
plt.title('2D Preview of Colour Grid')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 8. Assign Colours & Export

In [ ]:
if top_half is not None and bottom_half is not None:
    print("Assigning colors to split halves...")
    top_colors = assign_vertex_colors(top_half.vertices, tomo_lats, tomo_lons, tomo_grid, colormap=cmap, **colour_kwargs)
    bottom_colors = assign_vertex_colors(bottom_half.vertices, tomo_lats, tomo_lons, tomo_grid, colormap=cmap, **colour_kwargs)
    
    import os
    os.makedirs('../outputs', exist_ok=True)
    print("Exporting...")
    write_obj_with_vertex_colors('../outputs/tomo_globe_top.obj', top_half.vertices, top_half.faces, top_colors)
    write_obj_with_vertex_colors('../outputs/tomo_globe_bottom.obj', bottom_half.vertices, bottom_half.faces, bottom_colors)
    print("Exported split globes successfully.")
else:
    print("Skipping export — hemisphere creation failed.")

## 9. Visualise final globe in 3D

In [ ]:
import trimesh
import ipywidgets as widgets
from IPython.display import display

top_mesh = trimesh.Trimesh(vertices=top_half.vertices, faces=top_half.faces, vertex_colors=top_colors)
bottom_mesh = trimesh.Trimesh(vertices=bottom_half.vertices, faces=bottom_half.faces, vertex_colors=bottom_colors)

print('Upper mesh: (click and drag to rotate, mouse-scroll to zoom)')
display(top_mesh.show())
print('Lower mesh: (click and drag to rotate, mouse-scroll to zoom)')
display(bottom_mesh.show())